In [ ]:
from pathlib import Path
from PIL import Image
import copy
import cv2
import csv
import random
import numpy as np
from minerva.data.readers.png_reader import PNGReader
from minerva.data.readers.csv_reader import CSVReader
from minerva.data.readers.patched_array_reader import NumpyArrayReader
from minerva.data.data_modules.base import MinervaDataModule

from minerva.transforms.transform import _Transform, Identity
from minerva.transforms.random_transform import _RandomSyncedTransform, RandomFlip, RandomGrayScale, RandomSolarize, RandomRotation
from minerva.data.datasets.base import SimpleDataset

from minerva.utils.tensor import to_tensor

from minerva.models.ssl.byol import BYOL
from minerva.models.nets.image.deeplabv3 import DeepLabV3

from torchvision import transforms
import lightning as L
import torch
from torchmetrics import JaccardIndex
import matplotlib.pyplot as plt
import torchvision.models as models
from torch.utils.data import DataLoader
from torchmetrics.classification import MulticlassJaccardIndex
import torchvision.transforms.functional as F
#from torchvision.models import resnet50
import torch.nn as nn
from torchvision.models.resnet import resnet50
from lightning.pytorch.callbacks import Callback, ModelCheckpoint
from torchvision.models._utils import IntermediateLayerGetter


In [ ]:
class SimpleSegmentationHead(nn.Sequential):
    def __init__(self, in_channels, num_classes):
        super().__init__(
            nn.Conv2d(in_channels, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, num_classes, kernel_size=1)
        )

In [ ]:
class Identity_2(_Transform):
    """This class is a dummy transform that does nothing. It is useful when
    you want to skip a transform in a pipeline.
    """

    def __call__(self, x: np.ndarray) -> np.ndarray:
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        transform = transforms.Compose([transforms.ToTensor(), normalize])
        return transform(x)

    def __str__(self) -> str:
        return "Identity()"

In [ ]:
class Format_label_img(_Transform):

    def __call__(self, x: np.ndarray) -> torch.Tensor:

        if x.ndim == 3:
            x = x.squeeze()

        label_tensor = torch.from_numpy(x).long()

        if label_tensor.min() >= 1:
            label_tensor = label_tensor - 1

        return label_tensor

In [ ]:
def model_mIoU(model, trainer, dataloader):
    predictions = trainer.predict(model, dataloader)
    predictions = torch.cat(predictions, dim=0)
    pred_classes = torch.argmax(predictions, dim=1)
    test_samples = np.array([y for _, y in dataloader.test_dataset])
    y = torch.from_numpy(test_samples).long()
    jaccard = JaccardIndex(task="multiclass", num_classes=3)
    score = jaccard(pred_classes, y)
    #print(f"The mIoU of the model is {score.item()*100:.2f}%")
    return score.item()*100

In [ ]:
def average_miou(model, name, epochs):

    list_miou = []
    for i in range(5):
        _model = copy.deepcopy(model)

        # the path=Path('') is the path to the MATLAB segmentation dataset for finetuning 
        train_fine_data_reader = PNGReader(
            path=Path('dataset_spectogram/train/data')
        )
        train_fine_labels_reader = PNGReader(
            path=Path('dataset_spectogram/train/label')
        )
    
        train_fine_dataset = SimpleDataset(
            readers=[train_fine_data_reader, train_fine_labels_reader],
            transforms=[Identity_2(), Format_label_img()],
        )
    
        data_fine_module = MinervaDataModule(
            train_dataset=train_fine_dataset,
            val_dataset=val_fine_dataset,
            test_dataset=test_fine_dataset,
            batch_size=32,
            num_workers=0,
            #additional_train_dataloader_kwargs=False,
            name="simulated spectogram"
        )

        checkpoint_callback = ModelCheckpoint(
        save_top_k=-1,
        # save the checkpoint in path /checkpoints_finetuning, or you can change it, with name of the model and the iterantion i
        dirpath="checkpoints_finetuning",
        filename=f"{name}_{i}"
        )
    
        trainer_base = L.Trainer(
            max_epochs=epochs,
            accelerator="gpu",
            devices=1,
            logger=False,
            enable_checkpointing=False,
            callbacks=[checkpoint_callback]
        )
    
        trainer_base.fit(_model, data_fine_module)
    
        miou_score = model_mIoU(_model, _trainer, data_fine_module)

        list_miou.append(miou_score)
        
        del(_model)    

    return sum(list_miou) / len(list_miou)


In [ ]:
# the path=Path('') is the path to the MATLAB segmentation dataset for finetuning 

In [ ]:
val_fine_data_reader = PNGReader(
    path=Path('dataset_spectogram/val/data/')
)

In [ ]:
val_fine_labels_reader = PNGReader(
    path=Path('dataset_spectogram/val/label')
)

In [ ]:
test_fine_data_reader = PNGReader(
    path=Path('dataset_spectogram/test/data')
)

In [ ]:
test_fine_labels_reader = PNGReader(
    path=Path('dataset_spectogram/test/label')
)

In [ ]:
val_fine_dataset = SimpleDataset(
    readers=[val_fine_data_reader, val_fine_labels_reader],
    transforms=[Identity_2(), Format_label_img()],
)

print(val_fine_dataset)

In [ ]:
test_fine_dataset = SimpleDataset(
    readers=[test_fine_data_reader, test_fine_labels_reader],
    transforms=[Identity_2(), Format_label_img()],
)

print(test_fine_dataset)

In [ ]:
_trainer = L.Trainer(
    max_epochs=1,
    accelerator="gpu",
    devices=1,
    logger=False,
    enable_checkpointing=False
)

In [ ]:
RN50model_none_dilatation = resnet50(weights=None,replace_stride_with_dilation=[False, False, True])

In [ ]:
_RN50model_default_dilatation = torch.nn.Sequential(*list(RN50model_default_dilatation.children())[:-2])
model_default_dilatation_full = DeepLabV3(_RN50model_default_dilatation,learning_rate=1e-3,num_classes=3)

resnet_miou = average_miou(model_default_dilatation_full, "resnet50", 50)

In [ ]:
print(resnet_miou)

In [ ]:
# with learning rate 1e-3 the model could rarely overfit, use lr a little lower to be safe

_RN50model_none_dilatation = torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-2])
model_base_full = DeepLabV3(_RN50model_none_dilatation,learning_rate=1e-4,num_classes=3)

miou_base = average_miou(model_base_full, "base", 50)

In [ ]:
print(miou_base)

# config 1: VerticalLineMask(), AdditiveGaussianNoise(std=60)

## full seg_head

In [ ]:
# "checkpoins_byol/config_0_epochepoch=29.ckpt" should be the path to the byol checkpoint
byol_config_1 = BYOL.load_from_checkpoint("checkpoins_byol/config_1_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_default_dilatation.children())[:-1]), strict=True)
_backbone_config_1 = byol_config_1.backbone

In [ ]:
deeplab_backbone_1 = torch.nn.Sequential(*list(_backbone_config_1.children())[:-1])

In [ ]:
model_config_1 = DeepLabV3(deeplab_backbone_1,learning_rate=1e-3,num_classes=3)

In [ ]:
miou_1 = average_miou(model_config_1, "config_1", 50)

In [ ]:
print(miou_1)

# Config 2: HorizontalLineMask(), ColorJitter()

## full seg_head

In [ ]:
byol_config_2 = BYOL.load_from_checkpoint("checkpoins_byol/config_2_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_2 = byol_config_2.backbone

In [ ]:
deeplab_backbone_2 = torch.nn.Sequential(*list(_backbone_config_2.children())[:-1])

In [ ]:
model_config_2 = DeepLabV3(deeplab_backbone_2,learning_rate=1e-3,num_classes=3)

In [ ]:
miou_2 = average_miou(model_config_2, "config_2", 50)

In [ ]:
print(miou_2)

# Config 3: VerticalLineMask(), ColorJitter()

## full seg_head

In [ ]:
byol_config_3 = BYOL.load_from_checkpoint("checkpoins_byol/config_3_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_3 = byol_config_3.backbone

In [ ]:
deeplab_backbone_3 = torch.nn.Sequential(*list(_backbone_config_3.children())[:-1])

In [ ]:
model_config_3 = DeepLabV3(deeplab_backbone_3,learning_rate=1e-3,num_classes=3)

In [ ]:
miou_3 = average_miou(model_config_3, "config_3", 50)

In [ ]:
print(miou_3)

# Config 4: HorizontalLineMask(), AdditiveGaussianNoise()

## full seg_head

In [ ]:
byol_config_4 = BYOL.load_from_checkpoint("checkpoins_byol/config_4_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_4 = byol_config_4.backbone

In [ ]:
deeplab_backbone_4 = torch.nn.Sequential(*list(_backbone_config_4.children())[:-1])

In [ ]:
model_config_4 = DeepLabV3(deeplab_backbone_4,learning_rate=1e-3,num_classes=3)

In [ ]:
miou_4 = average_miou(model_config_4, "config_4", 50)

In [ ]:
print(miou_4)

# Config 5:
    1 - VerticalLineMask(), ColorJitter()
    2 - HorizontalLineMask(), AdditiveGaussianNoise()

## full seg_head

In [ ]:
byol_config_5 = BYOL.load_from_checkpoint("checkpoins_byol/config_5_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_5 = byol_config_5.backbone

In [ ]:
deeplab_backbone_5 = torch.nn.Sequential(*list(_backbone_config_5.children())[:-1])

In [ ]:
model_config_5 = DeepLabV3(deeplab_backbone_5,learning_rate=1e-3,num_classes=3)

In [ ]:
miou_5 = average_miou(model_config_5, "config_5", 50)

In [ ]:
print(miou_5)

# Config 6:
    1 - VerticalLineMask(), AdditiveGaussianNoise()
    2 - HorizontalLineMask(), ColorJitter()

## full seg_head

In [ ]:
byol_config_6 = BYOL.load_from_checkpoint("checkpoins_byol/config_6_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_6 = byol_config_6.backbone

In [ ]:
deeplab_backbone_6 = torch.nn.Sequential(*list(_backbone_config_6.children())[:-1])

In [ ]:
model_config_6 = DeepLabV3(deeplab_backbone_6,learning_rate=1e-3,num_classes=3)

In [ ]:
miou_6 = average_miou(model_config_6, "config_6", 50)

In [ ]:
print(miou_6)

# config 7:
    1 - VerticalLineMask(), ColorJitter()
    2 - HorizontalLineMask(), ColorJitter()

## full seg_head

In [ ]:
byol_config_7 = BYOL.load_from_checkpoint("checkpoins_byol/config_7_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_7 = byol_config_7.backbone

In [ ]:
deeplab_backbone_7 = torch.nn.Sequential(*list(_backbone_config_7.children())[:-1])

In [ ]:
model_config_7 = DeepLabV3(deeplab_backbone_7,learning_rate=1e-3,num_classes=3)

In [ ]:
miou_7 = average_miou(model_config_7, "config_7", 50)

In [ ]:
print(miou_7)

# config 8:
    1 - VerticalLineMask(), AdditiveGaussianNoise()
    2 - HorizontalLineMask(), AdditiveGaussianNoise()

## full seg_head

In [ ]:
byol_config_8 = BYOL.load_from_checkpoint("checkpoins_byol/config_8_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_config_8 = byol_config_8.backbone

In [ ]:
deeplab_backbone_8 = torch.nn.Sequential(*list(_backbone_config_8.children())[:-1])

In [ ]:
model_config_8 = DeepLabV3(deeplab_backbone_8,learning_rate=1e-3,num_classes=3)

In [ ]:
miou_8 = average_miou(model_config_8, "config_8", 50)

In [ ]:
print(miou_8)

# No dataset

## no panoradio

In [ ]:
byol_no_panoradio = BYOL.load_from_checkpoint("checkpoins_byol/no_panoradio_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_no_panoradio = byol_no_panoradio.backbone

In [ ]:
deeplab_backbone_no_panoradio = torch.nn.Sequential(*list(_backbone_no_panoradio.children())[:-1])

In [ ]:
model_no_panoradio = DeepLabV3(deeplab_backbone_no_panoradio,learning_rate=1e-3,num_classes=3)

In [ ]:
miou_no_panoradio = average_miou(model_no_panoradio, "no_panoradio", 50)

In [ ]:
print(miou_no_panoradio)

## no powder

In [ ]:
byol_no_powder = BYOL.load_from_checkpoint("checkpoins_byol/no_powder_epochepoch=29.ckpt", backbone=torch.nn.Sequential(*list(RN50model_none_dilatation.children())[:-1]), strict=True)
_backbone_no_powder = byol_no_powder.backbone

In [ ]:
deeplab_backbone_no_powder = torch.nn.Sequential(*list(_backbone_no_powder.children())[:-1])

In [ ]:
model_no_powder = DeepLabV3(deeplab_backbone_no_powder,learning_rate=1e-3,num_classes=3)

In [ ]:
miou_no_powder = average_miou(model_no_powder, "no_powder", 50)

In [ ]:
print(miou_no_powder)